In [1]:
import os
os.chdir('../../../../..')

In [2]:
from src.datasets import MaterialsProject

In [3]:
import numpy as np
import polars as pl

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN, KMeans
from kmedoids import KMedoids

from src.helper_functions import create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [4]:
mp = MaterialsProject(limit=5000, sampling_strategy="stratified", stratify_on=["band_gap", "energy_above_hull"], add_coulomb=True)
df = mp.load()

2026-04-21 14:18:00.067 | INFO     | src.datasets:load:1322 - Loading full cached Parquet data from data/Materials Project/materials.parquet...
2026-04-21 14:18:00.971 | INFO     | src.datasets:_add_descriptors:1671 - Ignoring output_tag=sample_n6000_seed40_stratified since descriptors are attached directly to dataframe.
2026-04-21 14:18:00.972 | INFO     | src.datasets:_add_descriptors:1705 - Calculating max atoms for Coulomb Matrix padding...
2026-04-21 14:18:01.256 | INFO     | src.datasets:_add_descriptors:1752 - Computing Coulomb Matrix chunk 0 (0 to 1000)...
2026-04-21 14:18:01.763 | INFO     | src.datasets:_add_descriptors:1752 - Computing Coulomb Matrix chunk 1 (1000 to 2000)...
2026-04-21 14:18:02.258 | INFO     | src.datasets:_add_descriptors:1752 - Computing Coulomb Matrix chunk 2 (2000 to 3000)...
2026-04-21 14:18:02.796 | INFO     | src.datasets:_add_descriptors:1752 - Computing Coulomb Matrix chunk 3 (3000 to 4000)...
2026-04-21 14:18:03.676 | INFO     | src.datasets:_add

In [5]:
dist_matrix = mp.get_distance_matrix(
    descriptor="coulomb",
    dist_type="euclidean",
    pca_variance=95,
    force_calculate=True,
)

2026-04-21 14:18:06.891 | INFO     | src.datasets:get_distance_matrix:1933 - Applying PCA to retain 95.00% of variance.
2026-04-21 14:18:42.414 | INFO     | src.datasets:get_distance_matrix:1942 - PCA reduced 'coulomb_matrix' dimensions from 10000 to 10
2026-04-21 14:18:42.433 | INFO     | src.datasets:get_distance_matrix:1953 - Calculating distance matrix for coulomb_matrix using euclidean distance.
2026-04-21 14:18:42.584 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/Materials Project/dist_coulomb_matrix_euclidean_pca0.95.npy


# Hierarchical Clustering on Distance Matrix

In [6]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=3, linkage='complete')
labels_hier = model_hier.fit_predict(dist_matrix)
print(np.unique(labels_hier, return_counts=True))
df = df.with_columns(labels_hier=labels_hier)

(array([0, 1, 2]), array([ 132,    3, 4865]))


In [7]:
average_numeric_by_cluster(df, "labels_hier")

shape: (3, 19)
┌─────────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬───────────┐
│ labels_hier ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a       ┆ b       ┆ c       ┆ alpha   ┆ beta    ┆ gamma   ┆ volume    ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ pct_metal │
│ ---         ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---       ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---       │
│ i64         ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64       ┆ f64       ┆ f64               ┆ f64  

labels_hier,count,max_en_diff,energy_per_atom,formation_energy_per_atom,band_gap,density,a,b,c,alpha,beta,gamma,volume,num_sites,energy_above_hull,avg_bond_length,max_bond_length,pct_metal
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,132,1.156212,-27.455127,-0.883035,0.23732,10.748582,8.60275,9.221167,11.824993,88.749987,88.857295,93.735867,787.155294,36.568182,0.032775,2.60136,2.77358,74.242424
1,3,1.076667,-6.977507,-1.041456,0.0,14.471273,10.548206,10.7723,16.185612,80.0,80.17541,90.0,1223.353808,73.333333,0.016619,2.447902,2.556311,100.0
2,4865,1.605998,-13.049132,-1.313,0.672307,6.077611,6.645421,7.081077,9.042861,86.035857,86.737711,88.544372,367.387076,21.978623,0.08313,2.289464,2.572538,58.232271


In [8]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="coulomb",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_hier,
    clustering_method="hierarchical"
)

2026-04-21 14:18:43.643 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/coulomb/pca_hierarchical_projection.png


{'coords': array([[2697726.16495534,  775134.52071925],
        [  39375.95691272, -340167.10652317],
        [-608319.89685587,  343663.50306851],
        ...,
        [-345752.62596215, -178330.17083463],
        [2206619.52091025,  397226.89582632],
        [-142499.22178026,  -89199.2631573 ]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/coulomb/pca_hierarchical_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/coulomb'),
 'clustering_method': 'hierarchical'}

In [9]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

# KMedoids

In [10]:
model_km = KMedoids(n_clusters=3, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
print(np.unique(labels_km, return_counts=True))
df = df.with_columns(labels_km=labels_km)

(array([0, 1, 2], dtype=uint64), array([1431, 1366, 2203]))


In [11]:
average_numeric_by_cluster(df, "labels_km")

shape: (3, 20)
┌───────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬─────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┐
│ labels_km ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c       ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ pct_metal │
│ ---       ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       │
│ u64       ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64      ┆ f64       ┆ f64

labels_km,count,max_en_diff,energy_per_atom,formation_energy_per_atom,band_gap,density,a,b,c,alpha,beta,gamma,volume,num_sites,energy_above_hull,avg_bond_length,max_bond_length,labels_hier,pct_metal
u64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1431,1.505486,-14.566277,-1.226863,0.529107,6.956795,6.076747,6.298469,8.209215,84.6219,84.837925,85.534088,245.729822,13.374563,0.081488,2.43425,2.700912,2.0,67.365479
1,1366,1.366223,-20.685859,-1.120703,0.492592,8.57494,7.503974,8.12942,10.238903,88.033523,88.561447,93.697971,519.221985,25.234993,0.03789,2.506357,2.747485,1.804539,67.203514
2,2203,1.792292,-8.183299,-1.462057,0.849781,4.249326,6.605052,7.072653,9.019177,85.870046,86.958986,87.617247,378.58197,26.49251,0.109139,2.079832,2.392697,2.0,47.753064


In [12]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

In [13]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="coulomb",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_km,
    clustering_method="kmedoids"
)

2026-04-21 14:18:48.866 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/coulomb/pca_kmedoids_projection.png


{'coords': array([[2697726.16495534,  775134.52071925],
        [  39375.95691272, -340167.10652317],
        [-608319.89685587,  343663.50306851],
        ...,
        [-345752.62596215, -178330.17083463],
        [2206619.52091025,  397226.89582632],
        [-142499.22178026,  -89199.2631573 ]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/coulomb/pca_kmedoids_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/coulomb'),
 'clustering_method': 'kmedoids'}

# Spectral

In [14]:
model_spectral = SpectralClustering(
                n_clusters=3,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)

KeyboardInterrupt: 

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="coulomb",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_spectral,
    clustering_method="spectral"
)

NameError: name 'labels_spectral' is not defined

In [ ]:
average_numeric_by_cluster(df, "labels_spectral")

shape: (3, 20)
┌─────────────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬────────┬─────────┬─────────┬──────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┐
│ labels_spectral ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c      ┆ alpha   ┆ beta    ┆ gamma    ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ labels_km │
│ ---             ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---      ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       │
│ i32             ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    ┆ f64    ┆ f64     ┆ f64     ┆ f64      ┆ f6

# DBSCAN

In [ ]:
model_db = DBSCAN(
    eps=7500,
    min_samples=5,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db,return_counts=True))

(array([-1,  0,  1,  2,  3,  4,  5]), array([ 172, 4756,   42,    7,    6,    8,    9]))


In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="coulomb",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_db,
    clustering_method="dbscan"
)

2026-04-19 14:06:06.701 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/coulomb/pca_dbscan_projection.png


{'coords': array([[2800442.63967447,  756369.6823298 ],
        [  48568.4980794 , -326827.80857034],
        [-609746.51810078,  345730.93704956],
        ...,
        [-355393.16155332, -171267.3980381 ],
        [2286565.81470132,  396153.62667518],
        [-157155.44091623,  -94446.57618272]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/coulomb/pca_dbscan_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/coulomb'),
 'clustering_method': 'dbscan'}

In [ ]:
average_numeric_by_cluster(df, "labels_db")

shape: (11, 22)
┌───────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────────┬──────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┬─────────────────┬───────────────────┐
│ labels_db ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a       ┆ b       ┆ c       ┆ alpha   ┆ beta     ┆ gamma    ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ labels_km ┆ labels_spectral ┆ labels_kmeans_raw │
│ ---       ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---      ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       ┆ ---             ┆ ---               │
│ i64       ┆ u32   ┆ f64         ┆ f64             ┆ f64   

# KMeans on Raw Embeddings

In [25]:
from sklearn.decomposition import PCA
X = np.array(df["coulomb_matrix"].to_list(), dtype=np.float32)
pca = PCA(n_components=4)
X = pca.fit_transform(X)

kmeans_raw = KMeans(n_clusters=6, random_state=42, n_init=10)
labels_kmeans_raw = kmeans_raw.fit_predict(X)
df = df.with_columns(labels_kmeans_raw=labels_kmeans_raw)

In [27]:
average_numeric_by_cluster(df, "labels_kmeans_raw")

shape: (6, 21)
┌───────────────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬─────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┬───────────┐
│ labels_kmeans_raw ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c       ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ labels_km ┆ pct_metal │
│ ---               ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       ┆ ---       │
│ i32               ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    

labels_kmeans_raw,count,max_en_diff,energy_per_atom,formation_energy_per_atom,band_gap,density,a,b,c,alpha,beta,gamma,volume,num_sites,energy_above_hull,avg_bond_length,max_bond_length,labels_hier,labels_km,pct_metal
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1127,1.462289,-15.357988,-1.195896,0.48172,7.250105,5.869343,6.061825,8.061484,84.158394,84.305335,84.582874,216.734188,11.972493,0.084957,2.443999,2.703292,2.0,0.017746,70.186335
1,315,1.231365,-25.292602,-0.964053,0.295952,9.933522,7.593197,8.270002,9.993003,90.602551,90.562946,93.824273,503.416783,25.149206,0.033063,2.569308,2.778839,1.695238,1.0,79.047619
2,2248,1.808581,-8.06312,-1.481597,0.845961,4.252105,6.492373,6.922955,8.788828,85.847216,86.86647,87.5408,353.139131,25.105427,0.111091,2.083013,2.39822,2.0,1.8621,48.309609
3,641,1.549251,-12.799116,-1.285209,0.767241,6.167187,8.118135,8.906263,11.254611,88.992097,89.963378,93.293617,655.599515,30.842434,0.037635,2.462927,2.743731,2.0,0.928237,49.765991
4,569,1.331371,-22.804023,-1.07688,0.443418,9.054328,6.596384,7.033204,8.968662,84.807013,85.36339,92.552455,318.123401,17.630931,0.046274,2.460624,2.701295,2.0,0.977153,72.58348
5,100,1.1684,-25.477963,-0.887861,0.201496,10.660682,9.386537,9.886112,12.952253,88.494598,88.950935,92.766832,969.686963,43.19,0.024547,2.638048,2.8027,0.29,1.0,76.0


In [26]:
create_chemiscope_viewer(df, X, labels_kmeans_raw, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…